In [ ]:
import pickle

with open("chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

In [ ]:
from langchain_core.documents import Document

docs = [Document(page_content=text) for text in all_chunks]

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

vdb = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="route"
)

In [ ]:
def dense_search(query, k=10):
    """Returns list of (doc, distance_score) using pure embedding similarity."""
    return vdb.similarity_search_with_score(query, k=k)

## Sparse Retrieval "BM25 keyword search"

In [ ]:
import re
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [simple_tokenize(doc.page_content) for doc in docs]
bm25 = BM25Okapi(tokenized_corpus)

def sparse_search(query, k=10):
    """Returns list of (doc, bm25_score) using keyword overlap."""
    scores = bm25.get_scores(simple_tokenize(query))
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [(docs[i], scores[i]) for i in ranked_idx]

In [ ]:
def add_scores(results, scores, rrf_k=60):
    for rank, (doc, _) in enumerate(results):
        scores[doc.page_content] = scores.get(doc.page_content, (0, doc))
        old_score, _ = scores[doc.page_content]
        scores[doc.page_content] = (old_score + 1 / (rrf_k + rank + 1), doc)

def reciprocal_rank_fusion(dense_results, sparse_results, k=10, rrf_k=60):
    scores = {}
    add_scores(dense_results, scores, rrf_k)
    add_scores(sparse_results, scores, rrf_k)

    ranked = sorted(scores.values(), key=lambda x: x[0], reverse=True)[:k]
    return [(doc, score) for score, doc in ranked]

def hybrid_search(query, k=10, pool=20):
    dense_results = dense_search(query, k=pool)
    sparse_results = sparse_search(query, k=pool)
    return reciprocal_rank_fusion(dense_results, sparse_results, k=k)

## Compare Dense-Only and Hybrid 

In [ ]:
test_queries = [
    "what is ML",             
    "overfitting",             
    "how does training a model work" 
]

for q in test_queries:
    print("=" * 80)
    print("QUERY:", q)
    print("-" * 80)
    print("DENSE-ONLY TOP RESULTS:")
    for doc, score in dense_search(q, k=3):
        print(f"  score={score:.4f} | {doc.page_content[:150].strip()}")
    print("-" * 80)
    print("HYBRID (RRF) TOP RESULTS:")
    for doc, score in hybrid_search(q, k=3):
        print(f"  rrf_score={score:.4f} | {doc.page_content[:150].strip()}")
    print()

## test

In [ ]:
query = "what is ML"

results = hybrid_search(query, k=3)

context = "\n\n".join(
    [doc.page_content for doc, score in results]
)
context

In [ ]:
prompt = f"""
You are an AI assistant in a Retrieval-Augmented Generation (RAG) system.

You must answer the user's question ONLY using the information provided in the retrieved context.

Instructions:
1. Use ONLY the retrieved context below.
2. Do NOT use your own knowledge.
3. If the answer is not contained in the context, reply exactly:
   "I cannot find the answer in the provided context."
4. Do not make up information or assumptions.
5. If multiple retrieved chunks contain relevant information, combine them into one coherent answer.
6. Keep the answer clear and concise.

======================
Retrieved Context:
{context}
======================

User Question:
{query}

Answer:
"""

In [ ]:
import google.generativeai as genai

genai.configure(api_key="your api key")

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(prompt)

print(response.text)